In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/vision_lab"
SEQUENCE = "structure_texture"

synced_dir = f"{BASE}/{SEQUENCE}/synced_data"
depth_gt_dir = f"{synced_dir}/depth"
depths_dir = f"{BASE}/{SEQUENCE}/depths"

import os
METHOD_NAMES = ["raw", "ssr_retinex", "lime", "clahe_std", "base_only", "reflectance_clahe", "hybrid"]
for name in METHOD_NAMES:
    n = len(os.listdir(f"{depths_dir}/{name}"))
    print(f"{name}: {n} depth maps found")

Mounted at /content/drive
raw: 106 depth maps found
ssr_retinex: 106 depth maps found
lime: 106 depth maps found
clahe_std: 106 depth maps found
base_only: 106 depth maps found
reflectance_clahe: 106 depth maps found
hybrid: 106 depth maps found


In [ ]:
import numpy as np
import cv2

MAX_DEPTH = 3.0  # locked in, matches your original DPT_Large 4-way cell

def scale_align(pred, gt, mask):
    # least-squares scale (your original lines 699-702) — the one we're keeping
    scale = np.sum(pred[mask] * gt[mask]) / (np.sum(pred[mask] ** 2) + 1e-8)
    return pred * scale

def mae(pred, gt, mask):
    return np.abs(pred[mask] - gt[mask]).mean()

def abs_rel(pred, gt, mask):
    return (np.abs(pred[mask] - gt[mask]) / gt[mask]).mean()

def rmse(pred, gt, mask):
    return np.sqrt(((pred[mask] - gt[mask]) ** 2).mean())

def delta_acc(pred, gt, mask, thresh=1.25):
    ratio = np.maximum(pred[mask] / gt[mask], gt[mask] / pred[mask])
    return (ratio < thresh).mean() * 100

In [ ]:
import time
t0 = time.time()
files = os.listdir(f"{depths_dir}/raw")
print(f"Found {len(files)} files in {time.time()-t0:.1f}s")

Found 106 files in 0.0s


In [ ]:
import shutil, time

local_depths_dir = "vision_workspace/depths_local"
t0 = time.time()
if not os.path.exists(local_depths_dir):
    shutil.copytree(depths_dir, local_depths_dir)
print(f"Copied depths to local disk in {time.time()-t0:.1f}s")

local_gt_dir = "vision_workspace/depth_gt_local"
if not os.path.exists(local_gt_dir):
    shutil.copytree(depth_gt_dir, local_gt_dir)
print(f"Copied ground truth to local disk in {time.time()-t0:.1f}s")

Copied depths to local disk in 32.9s
Copied ground truth to local disk in 77.7s


In [ ]:
from tqdm import tqdm

npy_files = sorted(os.listdir(f"{local_depths_dir}/raw"))
results = {name: [] for name in METHOD_NAMES}

for npy_fname in tqdm(npy_files, desc="Evaluating"):
    png_fname = npy_fname.replace(".npy", ".png")
    gt_path = os.path.join(local_gt_dir, png_fname)
    if not os.path.exists(gt_path):
        continue

    gt = cv2.imread(gt_path, cv2.IMREAD_UNCHANGED).astype(np.float32) / 5000.0
    mask = (gt > 0.1) & (gt < MAX_DEPTH)
    if mask.sum() == 0:
        continue

    for name in METHOD_NAMES:
        pred_path = f"{local_depths_dir}/{name}/{npy_fname}"
        if not os.path.exists(pred_path):
            continue
        pred = np.load(pred_path)
        pred_aligned = np.clip(scale_align(pred, gt, mask), 0.1, MAX_DEPTH)
        results[name].append((
            mae(pred_aligned, gt, mask),
            abs_rel(pred_aligned, gt, mask),
            rmse(pred_aligned, gt, mask),
            delta_acc(pred_aligned, gt, mask),
        ))

print(f"Evaluated {len(npy_files)} frames")

Evaluating: 100%|██████████| 106/106 [00:05<00:00, 17.98it/s]

Evaluated 106 frames


In [ ]:
import pandas as pd

rows = []
for name in METHOD_NAMES:
    r = np.array(results[name])
    rows.append({
        "method": name,
        "MAE": r[:, 0].mean(),
        "AbsRel": r[:, 1].mean(),
        "RMSE": r[:, 2].mean(),
        "delta_1.25": r[:, 3].mean(),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

df.to_csv(f"{BASE}/{SEQUENCE}/results.csv", index=False)
print(f"\nSaved to {BASE}/{SEQUENCE}/results.csv")

           method      MAE   AbsRel     RMSE  delta_1.25
              raw 0.203790 0.161811 0.242440   71.481697
      ssr_retinex 0.294292 0.225788 0.357934   54.485087
             lime 0.199049 0.157847 0.238570   72.280020
        clahe_std 0.214649 0.170730 0.265083   73.115029
        base_only 0.237507 0.182332 0.297348   65.478158
reflectance_clahe 0.216927 0.169882 0.270517   73.408980
           hybrid 0.276453 0.210731 0.346104   58.498154

Saved to /content/drive/MyDrive/vision_lab/structure_texture/results.csv


In [ ]:
# Test median scale_align vs least-squares on one frame
def scale_align_median(pred, gt, mask):
    scale = np.median(gt[mask]) / (np.median(pred[mask]) + 1e-8)
    return pred * scale

# Pick first frame
npy_fname = sorted(os.listdir(f"{depths_dir}/raw"))[0]
png_fname = npy_fname.replace(".npy", ".png")
gt = cv2.imread(f"{depth_gt_dir}/{png_fname}", cv2.IMREAD_UNCHANGED).astype(np.float32) / 5000.0
mask = (gt > 0.1) & (gt < 5.0)  # test with 5.0

raw_pred = np.load(f"{depths_dir}/raw/{npy_fname}")
hyb_pred = np.load(f"{depths_dir}/hybrid/{npy_fname}")

raw_ls  = np.clip(scale_align(raw_pred, gt, mask), 0.1, 5.0)
hyb_ls  = np.clip(scale_align(hyb_pred, gt, mask), 0.1, 5.0)
raw_med = np.clip(scale_align_median(raw_pred, gt, mask), 0.1, 5.0)
hyb_med = np.clip(scale_align_median(hyb_pred, gt, mask), 0.1, 5.0)

print("Least-squares align:")
print(f"  raw MAE={np.abs(raw_ls[mask]-gt[mask]).mean():.4f}  hybrid MAE={np.abs(hyb_ls[mask]-gt[mask]).mean():.4f}")
print("Median align:")
print(f"  raw MAE={np.abs(raw_med[mask]-gt[mask]).mean():.4f}  hybrid MAE={np.abs(hyb_med[mask]-gt[mask]).mean():.4f}")

Least-squares align:
  raw MAE=0.1569  hybrid MAE=0.4363
Median align:
  raw MAE=0.1498  hybrid MAE=0.3085


In [ ]:
import matplotlib.pyplot as plt

# Pick a textureless frame
fname = sorted(os.listdir("vision_workspace/synced/rgb_raw"))[50]
raw_img = cv2.imread(f"vision_workspace/synced/rgb_raw/{fname}")

# Reconstruct reflectance_clahe output visually
lab = cv2.cvtColor(raw_img, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
l_f = l.astype(np.float32)
illumination = cv2.bilateralFilter(l_f, d=15, sigmaColor=75, sigmaSpace=75)
reflectance = l_f - illumination
# the +128 shift
refl_shifted = np.clip(reflectance + 128, 0, 255).astype(np.uint8)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
refl_enhanced = clahe.apply(refl_shifted)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Raw")
axes[1].imshow(refl_shifted, cmap='gray')
axes[1].set_title("Reflectance+128 (what CLAHE sees)")
axes[2].imshow(refl_enhanced, cmap='gray')
axes[2].set_title("After CLAHE on reflectance")
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig("reflectance_clahe_check.png", dpi=150)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'vision_workspace/synced/rgb_raw'